In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re

In [2]:
df = pd.read_csv(r"..\data\raw\discord_messages_processed.csv")
df['Date'] = pd.to_datetime(df['Date'])
df.head()

,Author,Date,Content
0,A,2020-01-01 00:56:00.464000+07:00,server
1,A,2020-01-01 00:56:03.908000+07:00,will be oin overnight
2,B,2020-01-01 00:56:08.641000+07:00,awesome
3,A,2020-01-03 17:24:09.090000+07:00,you playing the craft?
4,A,2020-01-03 17:24:10.514000+07:00,server's on


In [3]:
# Generate (Message, Response) pairs
mr = pd.DataFrame(columns=['message', 'response'])
i = 0

while i < len(df) - 1:
	if df['Author'][i] == "A" or df['Author'][i + 1] == "B":
		i = i + 1
		continue
	
	m = str(df['Content'][i])
	r = str(df['Content'][i + 1])

	if '<URL>' in m or '<ATTACH>' in m or '<URL>' in r or '<ATTACH>' in r:
		i = i + 2
		continue

	if len(m) < 10 or len(r) < 10:
		i = i + 2
		continue

	if (df['Date'][i + 1] - df['Date'][i]).total_seconds() > 60:
		i = i + 2
		continue

	mr = pd.concat([mr, pd.DataFrame({'message': [m], 'response': [r]})], ignore_index=True)

	i = i + 2

len(mr)

11736

In [6]:
mr_sample = mr.sample(n=100, random_state=20)
train_sample, val_sample = train_test_split(mr_sample, test_size=0.2, random_state=42)
train_sample.to_csv(r'..\data\sample\mr_train.csv', index=False)
val_sample.to_csv(r'..\data\sample\mr_val.csv', index=False)

In [5]:
train_df, val_df = train_test_split(mr, test_size=0.2, random_state=42)
train_df.to_csv(r'..\data\processed\mr_train.csv', index=False)
val_df.to_csv(r'..\data\processed\mr_val.csv', index=False)